# Validação Cruzada (5-Fold)

Este notebook realiza a validação cruzada de 5 folds utilizando o modelo **BioBERTpt**

In [ ]:
!pip install -q transformers datasets peft evaluate seqeval accelerate scikit-learn

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import evaluate
from datasets import Dataset
from sklearn.model_selection import KFold
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    BertTokenizerFast
)
from peft import LoraConfig, get_peft_model, TaskType



In [ ]:
#Funções auxiliares
def carregar_arquivo_bio(caminho_arquivo):
    lista_dados = []
    tokens_atuais = []
    labels_atuais = []
    
    if not os.path.exists(caminho_arquivo):
        return []

    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        for linha in f:
            linha_limpa = linha.strip()
            if not linha_limpa:
                if len(tokens_atuais) > 0:
                    lista_dados.append({"tokens": tokens_atuais, "ner_tags": labels_atuais})
                    tokens_atuais = []
                    labels_atuais = []
                continue
            
            partes = linha_limpa.split()
            if len(partes) >= 2:
                tokens_atuais.append(partes[0])
                labels_atuais.append(partes[1])
                
    if len(tokens_atuais) > 0:
        lista_dados.append({"tokens": tokens_atuais, "ner_tags": labels_atuais})
        
    return lista_dados

def converter_rotulos_para_ids(exemplo):
    exemplo["ner_tags"] = [rotulo_para_id[r] for r in exemplo["ner_tags"]]
    return exemplo

def tokenizar_e_alinhar_labels(exemplos):
    entradas_tokenizadas = tokenizador(
        exemplos["tokens"], 
        truncation=True, 
        max_length=256, 
        is_split_into_words=True
    )

    lista_labels = []
    for i, labels in enumerate(exemplos["ner_tags"]):
        id_palavras = entradas_tokenizadas.word_ids(batch_index=i)
        indice_palavra_anterior = None
        id_labels_token = []
        
        for idx in id_palavras:
            if idx is None:
                id_labels_token.append(-100)
            elif idx != indice_palavra_anterior:
                id_labels_token.append(labels[idx])
            else:
                id_labels_token.append(-100)
            indice_palavra_anterior = idx
            
        lista_labels.append(id_labels_token)
        
    entradas_tokenizadas["labels"] = lista_labels
    return entradas_tokenizadas

#Função para Computação das Métricas com seqeval
avaliador_seqeval = evaluate.load("seqeval")

def calcular_metricas(predicoes_e_labels):
    from seqeval.metrics.sequence_labeling import get_entities
    predicoes, labels_reais = predicoes_e_labels
    predicoes = np.argmax(predicoes, axis=2)

    predicoes_verdadeiras = []
    labels_verdadeiros = []
    
    for predicao, label in zip(predicoes, labels_reais):
        linha_pred = []
        linha_label = []
        for p, l in zip(predicao, label):
            if l != -100:
                linha_pred.append(lista_rotulos[p])
                linha_label.append(lista_rotulos[l])
        predicoes_verdadeiras.append(linha_pred)
        labels_verdadeiros.append(linha_label)

    tp_global = 0
    fp_global = 0
    fn_global = 0
    
    tps_por_tipo = {}
    fps_por_tipo = {}
    fns_por_tipo = {}
    
    for seq_pred, seq_label in zip(predicoes_verdadeiras, labels_verdadeiros):
        preds_ent = get_entities(seq_pred)
        labels_ent = get_entities(seq_label)
        
        preds_decomposed = set()
        for tipo, inicio, fim in preds_ent:
            for sub_tipo in tipo.split("_"):
                preds_decomposed.add((sub_tipo, inicio, fim))
                
        labels_decomposed = set()
        for tipo, inicio, fim in labels_ent:
            for sub_tipo in tipo.split("_"):
                labels_decomposed.add((sub_tipo, inicio, fim))
                
        tps = preds_decomposed.intersection(labels_decomposed)
        fps = preds_decomposed - labels_decomposed
        fns = labels_decomposed - preds_decomposed
        
        tp_global += len(tps)
        fp_global += len(fps)
        fn_global += len(fns)
        
        for tipo, _, _ in tps:
            tps_por_tipo[tipo] = tps_por_tipo.get(tipo, 0) + 1
        for tipo, _, _ in fps:
            fps_por_tipo[tipo] = fps_por_tipo.get(tipo, 0) + 1
        for tipo, _, _ in fns:
            fns_por_tipo[tipo] = fns_por_tipo.get(tipo, 0) + 1
            
    tipos_basicos = set()
    for rotulo in lista_rotulos:
        if rotulo != "O":
            for sub_tipo in rotulo[2:].split("_"):
                tipos_basicos.add(sub_tipo)
                
    dicionario_metricas = {}
    for tipo in sorted(list(tipos_basicos)):
        tp = tps_por_tipo.get(tipo, 0)
        fp = fps_por_tipo.get(tipo, 0)
        fn = fns_por_tipo.get(tipo, 0)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        number = tp + fn
        
        dicionario_metricas[f"{tipo}_precision"] = precision
        dicionario_metricas[f"{tipo}_recall"] = recall
        dicionario_metricas[f"{tipo}_f1"] = f1
        dicionario_metricas[f"{tipo}_number"] = number
        
    precision_global = tp_global / (tp_global + fp_global) if (tp_global + fp_global) > 0 else 0.0
    recall_global = tp_global / (tp_global + fn_global) if (tp_global + fn_global) > 0 else 0.0
    f1_global = 2 * (precision_global * recall_global) / (precision_global + recall_global) if (precision_global + recall_global) > 0 else 0.0
    
    total_tokens = 0
    tokens_corretos = 0
    for seq_pred, seq_label in zip(predicoes_verdadeiras, labels_verdadeiros):
        for p, l in zip(seq_pred, seq_label):
            total_tokens += 1
            if p == l:
                tokens_corretos += 1
    accuracy_global = tokens_corretos / total_tokens if total_tokens > 0 else 0.0
    
    dicionario_metricas["precision"] = precision_global
    dicionario_metricas["recall"] = recall_global
    dicionario_metricas["f1"] = f1_global
    dicionario_metricas["accuracy"] = accuracy_global
    
    return dicionario_metricas


In [ ]:
#Identifica se está executando no Google Colab
esta_no_colab = 'google.colab' in sys.modules

if esta_no_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    diretorio_base = '/content/drive/MyDrive/ajuste-fino'
else:
    diretorio_base = '/kaggle/working/'

#Configurações do Modelo
checkpoint_modelo = "pucpr/biobertpt-all"
nome_base_experimento = "exp_cv_biobertpt"

#Configuração Geral 
caminho_dados_treino = os.path.join(diretorio_base, "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/train.bio")
caminho_dados_validacao = os.path.join(diretorio_base, "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/val.bio")
caminho_dados_teste = os.path.join(diretorio_base, "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/test.bio")
numero_folds = 5
numero_epocas = 5
semente = 42

#Cria o diretório de saída sequencial
contador = 1
diretorio_saida = os.path.join(diretorio_base, f"experimentos/{nome_base_experimento}_{contador}")
while os.path.exists(diretorio_saida):
    contador = contador + 1
    diretorio_saida = os.path.join(diretorio_base, f"experimentos/{nome_base_experimento}_{contador}")

os.makedirs(diretorio_saida, exist_ok=True)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizando hardware: {dispositivo}")


In [ ]:
#Junta os dados de treino e validação para fazer K-Fold
dados_brutos = carregar_arquivo_bio(caminho_dados_treino) + carregar_arquivo_bio(caminho_dados_validacao) + carregar_arquivo_bio(caminho_dados_teste)
print(f"Total de sentenças para validação cruzada: {len(dados_brutos)}")

#Cria o conjunto de todos os rótulos
conjunto_rotulos = set()
for item in dados_brutos:
    for rotulo in item["ner_tags"]:
        conjunto_rotulos.add(rotulo)
        
lista_rotulos = sorted(list(conjunto_rotulos))
rotulo_para_id = {}
id_para_rotulo = {}
for i, rotulo in enumerate(lista_rotulos):
    rotulo_para_id[rotulo] = i
    id_para_rotulo[i] = rotulo

#Inicializa o Tokenizador
tokenizador = BertTokenizerFast.from_pretrained(checkpoint_modelo)

#Função para Computação das Métricas com seqeval
avaliador_seqeval = evaluate.load("seqeval")



In [ ]:
#Loop de Validação Cruzada
from transformers import EarlyStoppingCallback

validador_kfold = KFold(n_splits=numero_folds, shuffle=True, random_state=semente)
historico_resultados = []

for fold, (indices_treino, indices_validacao) in enumerate(validador_kfold.split(dados_brutos)):
    print(f"\n==================== FOLD {fold+1}/{numero_folds} ====================")
    
    dados_treino_fold = []
    for idx in indices_treino:
        dados_treino_fold.append(dados_brutos[idx])
        
    dados_validacao_fold = []
    for idx in indices_validacao:
        dados_validacao_fold.append(dados_brutos[idx])
        
    dataset_treino = Dataset.from_list(dados_treino_fold).map(converter_rotulos_para_ids).map(tokenizar_e_alinhar_labels, batched=True, remove_columns=["tokens", "ner_tags"])
    dataset_validacao = Dataset.from_list(dados_validacao_fold).map(converter_rotulos_para_ids).map(tokenizar_e_alinhar_labels, batched=True, remove_columns=["tokens", "ner_tags"])
    
    modelo = AutoModelForTokenClassification.from_pretrained(
        checkpoint_modelo, 
        num_labels=len(lista_rotulos), 
        id2label=id_para_rotulo, 
        label2id=rotulo_para_id
    )
    
    configuracao_lora = LoraConfig(
        task_type=TaskType.TOKEN_CLS, 
        r=16, 
        lora_alpha=16, 
        lora_dropout=0.1, 
        target_modules=["query", "value"]
    )
    modelo_peft = get_peft_model(modelo, configuracao_lora)
    
    argumentos_treinamento = TrainingArguments(
        output_dir=os.path.join(diretorio_saida, f"fold_{fold+1}"),
        eval_strategy="epoch",
        learning_rate=3e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=numero_epocas,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        remove_unused_columns=True,
        report_to="none"
    )
    
    treinador = Trainer(
        model=modelo_peft,
        args=argumentos_treinamento,
        train_dataset=dataset_treino,
        eval_dataset=dataset_validacao,
        data_collator=DataCollatorForTokenClassification(tokenizador),
        compute_metrics=calcular_metricas,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    treinador.train()
    
    resultados_validacao = treinador.evaluate()
    print(f"Resultados Fold {fold+1}: {resultados_validacao}")
    historico_resultados.append(resultados_validacao)
    
    modelo_peft.save_pretrained(os.path.join(diretorio_saida, f"fold_{fold+1}", "model_final"))
    tokenizador.save_pretrained(os.path.join(diretorio_saida, f"fold_{fold+1}", "model_final"))

In [ ]:
#Relatório Final
print(f"\n==================== RELATÓRIO FINAL CROSS-VALIDATION ====================")
lista_f1 = []
for res in historico_resultados:
    lista_f1.append(res["eval_f1"])
    
media_f1 = np.mean(lista_f1)
desvio_padrao_f1 = np.std(lista_f1)
print(f"Média F1: {media_f1:.4f} (+/- {desvio_padrao_f1:.4f})")

#Identifica as categorias existentes no histórico
chaves_metricas = historico_resultados[0].keys()
categorias = []
for chave in chaves_metricas:
    if chave.startswith("eval_") and chave.endswith("_f1") and chave != "eval_f1":
        categoria = chave[5:-3]
        categorias.append(categoria)
categorias = sorted(categorias)

metricas_por_categoria = {}
for cat in categorias:
    lista_cat_f1 = [res[f"eval_{cat}_f1"] for res in historico_resultados if f"eval_{cat}_f1" in res]
    lista_cat_prec = [res[f"eval_{cat}_precision"] for res in historico_resultados if f"eval_{cat}_precision" in res]
    lista_cat_rec = [res[f"eval_{cat}_recall"] for res in historico_resultados if f"eval_{cat}_recall" in res]
    
    metricas_por_categoria[cat] = {
        "mean_f1": float(np.mean(lista_cat_f1)) if lista_cat_f1 else 0.0,
        "std_f1": float(np.std(lista_cat_f1)) if lista_cat_f1 else 0.0,
        "mean_precision": float(np.mean(lista_cat_prec)) if lista_cat_prec else 0.0,
        "std_precision": float(np.std(lista_cat_prec)) if lista_cat_prec else 0.0,
        "mean_recall": float(np.mean(lista_cat_rec)) if lista_cat_rec else 0.0,
        "std_recall": float(np.std(lista_cat_rec)) if lista_cat_rec else 0.0,
    }

print("\nMétricas por Categoria:")
for cat, met in metricas_por_categoria.items():
    print(f"  {cat}: F1={met['mean_f1']:.4f} (+/- {met['std_f1']:.4f}) | Precision={met['mean_precision']:.4f} | Recall={met['mean_recall']:.4f}")

#Salva os resultados finais em JSON
resultados_finais = {
    "model_checkpoint": checkpoint_modelo,
    "k_folds": numero_folds,
    "num_epochs": numero_epocas,
    "seed": semente,
    "average_f1": float(media_f1),
    "std_f1": float(desvio_padrao_f1),
    "category_metrics": metricas_por_categoria,
    "folds": historico_resultados
}

caminho_json_resultados = os.path.join(diretorio_saida, "cv_results.json")
with open(caminho_json_resultados, "w", encoding="utf-8") as f:
    json.dump(resultados_finais, f, indent=4, ensure_ascii=False)

#Salva relatório em TXT
caminho_relatorio = os.path.join(diretorio_saida, "cv_report.txt")
with open(caminho_relatorio, "w", encoding="utf-8") as f:
    f.write(f"RELATÓRIO FINAL CROSS-VALIDATION - {nome_base_experimento}\n")
    f.write(f"{'='*50}\n")
    f.write(f"Modelo: {checkpoint_modelo}\n")
    f.write(f"Épocas: {numero_epocas}\n")
    f.write(f"Folds: {numero_folds}\n")
    f.write(f"Parâmetros do LoRA:\n")
    f.write(f"  r: {configuracao_lora.r}\n")
    f.write(f"  lora_alpha: {configuracao_lora.lora_alpha}\n")
    f.write(f"  lora_dropout: {configuracao_lora.lora_dropout}\n")
    f.write(f"  target_modules: {configuracao_lora.target_modules}\n")
    f.write(f"Parâmetros de Treinamento:\n")
    f.write(f"  Learning Rate: {argumentos_treinamento.learning_rate}\n")
    f.write(f"  Batch Size Treino: {argumentos_treinamento.per_device_train_batch_size}\n")
    f.write(f"  Batch Size Eval: {argumentos_treinamento.per_device_eval_batch_size}\n")
    f.write(f"  Weight Decay: {argumentos_treinamento.weight_decay}\n")
    f.write(f"  Semente: {semente}\n")
    f.write(f"{'='*50}\n")
    f.write(f"Média F1 Geral: {media_f1:.4f} (+/- {desvio_padrao_f1:.4f})\n\n")
    
    f.write(f"Métricas Médias por Categoria:\n")
    for cat, met in metricas_por_categoria.items():
        f.write(f"  {cat}:\n")
        f.write(f"    F1: {met['mean_f1']:.4f} (+/- {met['std_f1']:.4f})\n")
        f.write(f"    Precision: {met['mean_precision']:.4f} (+/- {met['std_precision']:.4f})\n")
        f.write(f"    Recall: {met['mean_recall']:.4f} (+/- {met['std_recall']:.4f})\n")
    f.write(f"\nDetalhes por Fold:\n")
    for i, res in enumerate(historico_resultados):
        f.write(f"Fold {i+1}: F1={res['eval_f1']:.4f}, Precision={res['eval_precision']:.4f}, Recall={res['eval_recall']:.4f}, Accuracy={res['eval_accuracy']:.4f}\n")
        for cat in categorias:
            f_key = f"eval_{cat}_f1"
            p_key = f"eval_{cat}_precision"
            r_key = f"eval_{cat}_recall"
            if f_key in res:
                f.write(f"  - {cat}: F1={res[f_key]:.4f}, Precision={res[p_key]:.4f}, Recall={res[r_key]:.4f}\n")

print(f"\nResultados salvos em: {diretorio_saida}")
